In [1]:
import pandas as pd
import numpy as np
from datetime import datetime


In [2]:
# Load the dataset
df = pd.read_csv('Superstore.csv', encoding='cp1252')
# Drop unnecessary columns
df.drop(columns=['Row ID', 'Order ID', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'Postal Code', 'Product ID'], inplace=True)
df.head()

,Order Date,City,State,Region,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,11/8/2016,Henderson,Kentucky,South,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,11/8/2016,Henderson,Kentucky,South,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,6/12/2016,Los Angeles,California,West,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,10/11/2015,Fort Lauderdale,Florida,South,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,10/11/2015,Fort Lauderdale,Florida,South,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
# Convert each transaction to a natural language description
document = []
for index, row in df.iterrows():
    date_string = datetime.strptime(row['Order Date'],'%m/%d/%Y').strftime('%B %d, %Y')
    quantity_string = f"{row['Quantity']} units"
    product_string = f"{row['Product Name']} ({row['Category']} - {row['Sub-Category']})"
    discount_string = f" with a discount of {row['Discount']*100} percent" if row['Discount'] > 0 else ""
    place_string = f"in {row['City']}, {row['State']} ({row['Region']})"
    string = f"On {date_string}, {quantity_string} of {product_string} were sold {place_string} for {row['Sales']} dollars{discount_string}, resulting in a profit of {row['Profit']} dollars."
    document.append(string)

In [4]:
# Create aggregated summaries
df['Order Date'] = pd.to_datetime(df['Order Date'])
monthly_sales = df.groupby(df['Order Date'].dt.to_period('M'))['Sales'].sum()
category_performance = df.groupby('Category')['Sales'].sum()
regional_analysis = df.groupby('Region')['Sales'].sum()
print(f"Monthly Sales: {monthly_sales.sum()}")
print(f"Category Performance: {category_performance.to_dict()}")
print(f"Regional Analysis: {regional_analysis.to_dict()}")

Monthly Sales: 2297200.8603
Category Performance: {'Furniture': 741999.7953, 'Office Supplies': 719047.032, 'Technology': 836154.033}
Regional Analysis: {'Central': 501239.8908, 'East': 678781.24, 'South': 391721.905, 'West': 725457.8245}


In [6]:
# Create statistical summaries as txt files
with open('statistical_summaries.txt', 'w') as f:
    f.write(f"Total Sales: {df['Sales'].sum()}\n")
    f.write(f"Total Profit: {df['Profit'].sum()}\n")
    f.write(f"Average Discount: {df['Discount'].mean()}\n")
    f.write(f"Average Quantity Sold: {df['Quantity'].mean()}\n")
    f.write(f"Average Sales per Order: {df['Sales'].mean()}\n")
    f.write(f"Average Profit per Order: {df['Profit'].mean()}\n")
    f.write(f"Total Orders: {len(df)}\n")


In [7]:
# Experiment with different chunk sizes for the RAG model
chunk_sizes = [500, 1000, 2000]
def chunk_document(document, chunk_size):
    return [document[i:i + chunk_size] for i in range(0, len(document), chunk_size)]